# 05 — Survival Modelling

**Purpose:** Train and persist all survival models defined in `config/features.yaml`.

Models:
- **Cox Proportional Hazards** (lifelines) — interpretable baseline
- **Random Survival Forest** (scikit-survival) — non-linear, handles interactions
- **Gradient Boosting Survival** (scikit-survival) — best performance, less interpretable

Cross-validation follows the Harrell C-index on each fold to compare before committing to
a held-out test set.

---
**Inputs:** `data/processed/X_train.parquet`, `y_train.parquet`, `selected_features.json`  
**Outputs:** `models/cox_model.pkl`, `models/rsf_model.pkl`, `models/gbs_model.pkl`

In [ ]:
import sys, json, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sksurv.util import Surv
from sksurv.metrics import concordance_index_censored

from src.data_utils import load_data, load_config, DATA_PROCESSED
from src.survival_models import build_models_from_config, CoxModel, RSFModel, GBSModel

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 40)
print('Ready.')

In [ ]:
cfg = load_config()
DURATION_COL = cfg['target']['duration_col']
EVENT_COL    = cfg['target']['event_col']
SEED         = cfg['experiment']['random_state']
CV_FOLDS     = cfg['experiment']['cv_folds']

X_train = load_data('X_train', stage='processed')
y_train = load_data('y_train', stage='processed')

duration_train = y_train[DURATION_COL]
event_train    = y_train[EVENT_COL]

# Load selected features from notebook 04
sel_path = DATA_PROCESSED / 'selected_features.json'
if sel_path.exists():
    with open(sel_path) as f:
        sel_info = json.load(f)
    SELECTED = sel_info['selected_features']
    print(f'Using {len(SELECTED)} selected features: {SELECTED}')
else:
    SELECTED = list(X_train.columns)
    print(f'selected_features.json not found — using all {len(SELECTED)} features.')

X_train_sel = X_train[SELECTED]

## 1. Cross-Validation

Evaluate each model with stratified K-fold CV before final fit on full training data.

In [ ]:
def cv_cindex(model_class, model_kwargs, X, duration, event, n_folds=5, seed=42):
    """Return mean and std C-index across K folds."""
    kf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    scores = []
    for fold, (tr_idx, val_idx) in enumerate(kf.split(X, event)):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        dur_tr, dur_val = duration.iloc[tr_idx], duration.iloc[val_idx]
        ev_tr,  ev_val  = event.iloc[tr_idx],    event.iloc[val_idx]

        mdl = model_class(**model_kwargs)
        mdl.fit(X_tr, dur_tr, ev_tr)
        risk = mdl.predict_risk(X_val)
        c, *_ = concordance_index_censored(ev_val.astype(bool).values, dur_val.values, risk)
        scores.append(c)
        print(f'  Fold {fold+1}: C-index = {c:.4f}')

    return np.mean(scores), np.std(scores)

print('Cross-validation will run below...')

In [ ]:
cv_results = {}

mcfg = cfg.get('models', {})

# Cox PH
if mcfg.get('cox_ph', {}).get('enabled', True):
    print('\n── Cox PH CV ──')
    c = mcfg['cox_ph']
    mean_c, std_c = cv_cindex(
        CoxModel, {'penalizer': c.get('penalizer', 0.1)},
        X_train_sel, duration_train, event_train, CV_FOLDS, SEED
    )
    cv_results['cox_ph'] = {'mean_cindex': round(mean_c, 4), 'std': round(std_c, 4)}
    print(f'  → Mean C-index = {mean_c:.4f} ± {std_c:.4f}')

# RSF
if mcfg.get('random_survival_forest', {}).get('enabled', True):
    print('\n── Random Survival Forest CV ──')
    c = mcfg['random_survival_forest']
    mean_c, std_c = cv_cindex(
        RSFModel,
        {'n_estimators': 50, 'min_samples_leaf': c.get('min_samples_leaf', 5), 'random_state': SEED},
        X_train_sel, duration_train, event_train, CV_FOLDS, SEED
    )
    cv_results['rsf'] = {'mean_cindex': round(mean_c, 4), 'std': round(std_c, 4)}
    print(f'  → Mean C-index = {mean_c:.4f} ± {std_c:.4f}')

# GBS
if mcfg.get('gradient_boosting_survival', {}).get('enabled', True):
    print('\n── Gradient Boosting Survival CV ──')
    c = mcfg['gradient_boosting_survival']
    mean_c, std_c = cv_cindex(
        GBSModel,
        {'n_estimators': 50, 'learning_rate': c.get('learning_rate', 0.05), 'random_state': SEED},
        X_train_sel, duration_train, event_train, CV_FOLDS, SEED
    )
    cv_results['gbs'] = {'mean_cindex': round(mean_c, 4), 'std': round(std_c, 4)}
    print(f'  → Mean C-index = {mean_c:.4f} ± {std_c:.4f}')

cv_df = pd.DataFrame(cv_results).T
print('\nCV Summary:')
display(cv_df.sort_values('mean_cindex', ascending=False))

## 2. Final Training on Full Training Set

In [ ]:
# Instantiate all enabled models from config
models = build_models_from_config(cfg)

print(f'Training {len(models)} models on {X_train_sel.shape}...')
for name, model in models.items():
    print(f'  Fitting {name}...')
    model.fit(X_train_sel, duration_train, event_train)
    print(f'  ✓ {name} fitted')

## 3. Cox PH — Coefficient Interpretation

In [ ]:
cox = models.get('cox_ph')
if cox:
    summary = cox.summary()
    print('Cox PH Summary:')
    display(summary[['coef', 'exp(coef)', 'p', 'coef lower 95%', 'coef upper 95%']].round(4))

    # Forest plot of hazard ratios
    coefs = summary['exp(coef)']
    ci_low  = summary['exp(coef) lower 95%']
    ci_high = summary['exp(coef) upper 95%']

    fig, ax = plt.subplots(figsize=(8, max(4, len(coefs) * 0.4)))
    y_pos = range(len(coefs))
    ax.errorbar(
        coefs.values, y_pos,
        xerr=[coefs.values - ci_low.values, ci_high.values - coefs.values],
        fmt='o', color='steelblue', capsize=4, linewidth=1.5,
    )
    ax.axvline(1.0, color='tomato', linestyle='--', label='HR = 1 (no effect)')
    ax.set_yticks(list(y_pos))
    ax.set_yticklabels(coefs.index)
    ax.set_xlabel('Hazard Ratio (exp[coef])')
    ax.set_title('Cox PH — Hazard Ratios with 95% CI')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 4. Tree Model Feature Importances

In [ ]:
from src.evaluation import plot_feature_importance

for name in ['rsf', 'gbs']:
    mdl = models.get(name)
    if mdl:
        imp = mdl.feature_importances()
        plot_feature_importance(imp, title=f'{name.upper()} Feature Importance', top_n=15)

## 5. Save Models

In [ ]:
for name, model in models.items():
    model.save(name)

# Save selected features alongside models for reproducibility
import json
from src.data_utils import MODELS_DIR
with open(MODELS_DIR / 'training_features.json', 'w') as f:
    json.dump({'features': SELECTED, 'n': len(SELECTED)}, f, indent=2)
print('\n✓ All models saved. Proceed to 06_model_evaluation.ipynb')